# Phase 6 — LoRA finetuning of Stable Diffusion 1.5

Trains rank-16 LoRA adapters on UNet AND text encoder using the 813 (image, caption) pairs from train.csv. Generates validation snapshots every 100 steps to monitor progress visually.

**Runtime:** ~30-40 min on A100. The 6 validation snapshots add ~5 min total.

**Output:**
- `lora_unet/` and `lora_text_encoder/` — adapter weights
- `snapshots/` — 6-image grids at each validation step
- `train_log.csv` — loss per step
- `config.json` — full hyperparameter snapshot

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"

import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR/projects/challenges-in-attribute-control

## 2. Install deps (diffusers + peft are the heavy additions)

In [ ]:
!pip install -q uv
!uv pip install -e . --system
!pip install -q diffusers accelerate peft transformers

## 3. HF login (SD 1.5 needs accepting its license)

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm A100

In [ ]:
import torch
assert torch.cuda.is_available()
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name} ({vram:.1f} GB)')
assert vram >= 35, 'A100 needed for SD 1.5 + LoRA on both UNet+text encoder'

## 5. Prepare train.csv with absolute paths

The train.csv from Phase 5 has paths relative to where it was generated. We rewrite them to point at the Drive locations.

In [ ]:
TRAIN_DRIVE = '/content/drive/MyDrive/binding-research/finetuning/train.csv'
import os; assert os.path.exists(TRAIN_DRIVE), f'No train.csv at {TRAIN_DRIVE}'
import pandas as pd
df = pd.read_csv(TRAIN_DRIVE)
print(f'Train CSV: {len(df)} rows, {df.image_path.nunique()} unique images')
print(f'Sample path: {df.image_path.iloc[0]}')

assert os.path.exists(df.image_path.iloc[0]), 'First image not accessible — paths may need rewriting'

## 6. Run LoRA training

Saves output to `/content/lora_output/`. ~30-40 min.

In [ ]:
!python experiments/exp6_lora_train.py \
    --train-csv '$TRAIN_DRIVE' \
    --output-dir /content/lora_output \
    --epochs 10 \
    --batch-size 4 \
    --grad-accum 2 \
    --learning-rate 1e-4 \
    --val-every 100 \
    --lora-rank 16

## 7. View validation snapshots (the training story in pictures)

Each snapshot is a 2x3 grid: top row = 3 treated prompts, bottom row = 2 control prompts + 1 more treated. Comparing snapshot_00100 (early) vs snapshot_01000 (final) shows the trajectory.

In [ ]:
from PIL import Image
from IPython.display import display
import os

snaps = sorted(os.listdir('/content/lora_output/snapshots'))
print(f'{len(snaps)} snapshots saved')

for fn in [snaps[0], snaps[len(snaps)//2], snaps[-1]]:
    print(f'\n=== {fn} ===')
    display(Image.open(os.path.join('/content/lora_output/snapshots', fn)))

## 8. Plot loss curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
log = pd.read_csv('/content/lora_output/train_log.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(log['step'], log['loss'], alpha=0.4, label='raw')
ax1.plot(log['step'], log['loss'].rolling(20).mean(), color='red', label='rolling mean (20)')
ax1.set_xlabel('step'); ax1.set_ylabel('MSE loss'); ax1.set_title('Training loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(log['step'], log['lr'])
ax2.set_xlabel('step'); ax2.set_ylabel('LR'); ax2.set_title('Learning rate (warmup + cosine)')
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Save everything to Drive

In [ ]:
import shutil
DEST = '/content/drive/MyDrive/binding-research/lora_output'
shutil.copytree('/content/lora_output', DEST, dirs_exist_ok=True)
print(f'Saved to {DEST}')